# 03 — 90-day and 180-day retention

Use first registrations from 2015–2018 and the same estimated entry-age groups
as notebook 02. Outcomes can extend into 2019. A member is retained at a horizon
if no final exit is recorded on or before that day after joining.


In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/raw.xlsx").is_file()
             and (p / "src/analysis.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.pipeline import load_data, preparation_tables, monthly_tables, save_tables

raw, prepared, members = load_data()
OUT = ROOT / "results"

from src.analysis import retention_results


For each joining month/year, age group and horizon, count eligible
members and retained members. Early known exits remain in the denominator.
Records censored before the horizon are excluded. All selected 2015–2018
entrants have sufficient potential follow-up for 180 days.


In [2]:
tables = retention_results(members)
tables["CohortRetention"][["CohortMonth", "AgeGroup", "HorizonDays",
                           "EligibleN", "RetainedN", "RetentionRate"]].head(8)

,CohortMonth,AgeGroup,HorizonDays,EligibleN,RetainedN,RetentionRate
0,2015-01-01,A: entry age 18-24,90,88,88,1.000000
1,2015-01-01,B: entry age 35+,90,44,44,1.000000
2,2015-02-01,A: entry age 18-24,90,157,157,1.000000
3,2015-02-01,B: entry age 35+,90,36,36,1.000000
4,2015-03-01,A: entry age 18-24,90,372,371,0.997312
5,2015-03-01,B: entry age 35+,90,55,55,1.000000
6,2015-04-01,A: entry age 18-24,90,171,170,0.994152
7,2015-04-01,B: entry age 35+,90,47,47,1.000000


Raw rates divide retained members by eligible members. Adjusted
rates give both age groups the same mix of joining cohorts. For each cohort,
the weight is its combined eligible A+B count divided by the combined eligible
count across the selected cohorts. `MatchedRateA/B` are the resulting weighted
averages. Calculations are in `retention_cohorts` and `retention_summary`.


In [3]:
summary = tables["Summary"].query("Scope == 'All 2015-2018'")
view = summary[["HorizonDays", "EligibleNA", "EligibleNB", "MatchedRateA", "MatchedRateB"]].copy()
view[["MatchedRateA", "MatchedRateB"]] *= 100
view = view.rename(columns={"MatchedRateA": "18–24 (%)", "MatchedRateB": "35+ (%)"})
view.round(2)

,HorizonDays,EligibleNA,EligibleNB,18–24 (%),35+ (%)
0,90,4899,2118,78.79,86.69
9,180,4899,2118,57.59,67.01


The sensitivity table varies age boundaries, final-day handling,
minimum cohort size and cohort weights. These are descriptive comparisons of
recorded final exits, not uninterrupted payment histories.


In [4]:
tables["Sensitivity"][["Scope", "HorizonDays", "MatchedRateA",
                       "MatchedRateB", "GapPP_AminusB"]].round(4)

,Scope,HorizonDays,MatchedRateA,MatchedRateB,GapPP_AminusB
0,Lower entry-age bound,90,0.7824,0.8726,-9.0173
1,Lower entry-age bound,180,0.5590,0.6790,-11.9996
2,Upper entry-age bound,90,0.7841,0.8700,-8.5831
3,Upper entry-age bound,180,0.5689,0.6781,-10.9231
4,Stable age assignments only,90,0.7830,0.8727,-8.9677
5,Stable age assignments only,180,0.5617,0.6785,-11.6781
6,Final day included,90,0.7954,0.8708,-7.5397
7,Final day included,180,0.5774,0.6715,-9.4103
8,Both groups N >= 20,90,0.7907,0.8695,-7.8822
9,Equal weight per exact cohort,90,0.7567,0.8379,-8.1245


Save all joining-cohort and summary tables. In Tableau, select one
`Scope` and keep horizons separate. For the main chart use `All 2015-2018` and
`MatchedRateA/B`, formatted as percentages.


In [5]:
save_tables(OUT / "retention", tables)
pd.DataFrame({"Saved table": list(tables)})

,Saved table
0,Summary
1,CohortRetention
2,CohortComparisons
3,Sensitivity
